# MATH 5010 Computer Lab — Section 9  
## Convergence Theory and Limit Theorems  
### Full Solutions Included

This lab accompanies **Section 9: Convergence Theory and Limit Theorems**.

We will use Python to study:

1. Convergence in distribution  
2. Convergence in probability  
3. Convergence in mean  
4. Almost sure convergence  
5. Relationships among convergence modes  
6. Weak Law of Large Numbers  
7. Strong Law of Large Numbers  
8. Central Limit Theorem  
9. Continuous Mapping Theorem  
10. Slutsky's Theorem  
11. Concentration inequalities  
12. Chebyshev versus Hoeffding rates  
13. High-dimensional proportion estimation  
14. Practice problems with complete solutions

The main message is that limit theorems explain why averages stabilize, why normal approximations appear, and how statistical estimators behave as sample size grows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from math import sqrt, exp, log

rng = np.random.default_rng(5010)

pd.set_option("display.precision", 5)
print("Packages loaded.")

## 1. Four Modes of Convergence

Let $X_n$ and $X$ be random variables.

### Convergence in probability

\[
X_n \xrightarrow{P} X
\]

means for every $\epsilon>0$,

\[
P(|X_n-X|>\epsilon)\to 0.
\]

### Convergence in distribution

\[
X_n \xrightarrow{D} X
\]

means

\[
F_{X_n}(x)\to F_X(x)
\]

at every continuity point $x$ of $F_X$.

### Convergence in mean

\[
X_n \xrightarrow{L^1} X
\]

means

\[
E|X_n-X|\to 0.
\]

### Almost sure convergence

\[
X_n\xrightarrow{a.s.} X
\]

means

\[
P\left(\lim_{n\to\infty}X_n=X\right)=1.
\]

## 2. Convergence in Probability: A Direct Example

Define

\[
X_n=
\begin{cases}
n, & \text{with probability }1/n^2,\\
0, & \text{with probability }1-1/n^2.
\end{cases}
\]

Then for any fixed $\epsilon>0$ and large enough $n$,

\[
P(|X_n-0|>\epsilon)=P(X_n=n)=\frac1{n^2}\to 0.
\]

Therefore,

\[
X_n\xrightarrow{P}0.
\]

But

\[
E[X_n]=n\cdot \frac1{n^2}=\frac1n\to 0.
\]

A related example with $X_n=n^2$ with probability $1/n$ converges in probability to $0$ but has diverging expectation.

In [ ]:
n_values = np.arange(1, 501)
eps = 0.5

prob_tail = 1 / n_values**2
expectation = 1 / n_values

plt.figure(figsize=(7, 4))
plt.plot(n_values, prob_tail, label=r"$P(|X_n|>\epsilon)=1/n^2$")
plt.plot(n_values, expectation, label=r"$E[X_n]=1/n$")
plt.xlabel("n")
plt.ylabel("value")
plt.title("Convergence in Probability and Mean")
plt.legend()
plt.show()

### Full Solution

For any $\epsilon>0$, when $n>\epsilon$,

\[
P(|X_n|>\epsilon)=P(X_n=n)=\frac1{n^2}.
\]

Since $1/n^2\to 0$,

\[
X_n\xrightarrow{P}0.
\]

Also,

\[
E|X_n-0|
=
E[X_n]
=
n\cdot \frac1{n^2}
=
\frac1n\to0.
\]

So in this example, $X_n\to0$ both in probability and in mean.

## 3. Convergence in Probability Without Convergence in Mean

Now define

\[
X_n=
\begin{cases}
n^2, & \text{with probability }1/n,\\
0, & \text{with probability }1-1/n.
\end{cases}
\]

Then

\[
P(|X_n|>\epsilon)=\frac1n\to0,
\]

so $X_n\xrightarrow{P}0$. But

\[
E[X_n]=n^2\cdot \frac1n=n\to\infty.
\]

Thus convergence in probability does not imply convergence in mean.

In [ ]:
n_values = np.arange(1, 501)
prob_tail = 1/n_values
expectation = n_values

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(n_values, prob_tail, label=r"$P(|X_n|>\epsilon)=1/n$")
ax1.set_xlabel("n")
ax1.set_ylabel("tail probability")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(n_values, expectation, linestyle="--", label=r"$E[X_n]=n$")
ax2.set_ylabel("expectation")
ax2.legend(loc="upper right")

plt.title("Convergence in Probability but Diverging Expectation")
plt.show()

### Full Solution

For any fixed $\epsilon>0$ and sufficiently large $n$,

\[
P(|X_n|>\epsilon)=P(X_n=n^2)=\frac1n\to0.
\]

Therefore,

\[
X_n\xrightarrow{P}0.
\]

However,

\[
E[X_n]=n^2\cdot\frac1n=n,
\]

which diverges. Hence $X_n$ does not converge to $0$ in mean.

## 4. Convergence in Distribution: Bernoulli Example

Let $X_n\sim \mathrm{Bernoulli}(p_n)$, where

\[
p_n=\frac12+\frac1n.
\]

Let

\[
X\sim \mathrm{Bernoulli}\left(\frac12\right).
\]

Then

\[
X_n\xrightarrow{D}X.
\]

We verify this by checking CDF convergence.

In [ ]:
def bernoulli_cdf(t, p):
    if t < 0:
        return 0.0
    elif t < 1:
        return 1 - p
    else:
        return 1.0

t_grid = [-0.5, 0.0, 0.5, 1.5]
n_values = [2, 5, 10, 50, 200, 1000]

rows = []
for t in t_grid:
    for n in n_values:
        p_n = 0.5 + 1/n
        rows.append([t, n, bernoulli_cdf(t, p_n), bernoulli_cdf(t, 0.5)])

df = pd.DataFrame(rows, columns=["t", "n", "F_Xn(t)", "F_X(t)"])
display(df)

### Full Solution

For $0\le t<1$,

\[
F_{X_n}(t)=P(X_n=0)=1-p_n
=
\frac12-\frac1n
\to \frac12.
\]

For $t<0$,

\[
F_{X_n}(t)=0=F_X(t).
\]

For $t\ge1$,

\[
F_{X_n}(t)=1=F_X(t).
\]

At continuity points of the Bernoulli$(1/2)$ CDF, $F_{X_n}(t)\to F_X(t)$, so

\[
X_n\xrightarrow{D}X.
\]

## 5. Almost Sure Convergence: Sample Paths

Let $U_1,U_2,\ldots$ be iid Uniform$(0,1)$ and define

\[
X_n=U_n^n.
\]

For any $\epsilon>0$,

\[
P(X_n>\epsilon)=P(U_n>\epsilon^{1/n})=1-\epsilon^{1/n}.
\]

For large $n$,

\[
1-\epsilon^{1/n}\approx \frac{-\log \epsilon}{n},
\]

so $\sum_n P(X_n>\epsilon)=\infty$. This example is not convenient for almost sure convergence.

Instead, define

\[
X_n=U_n/n.
\]

Then

\[
0\le X_n\le \frac1n\to0
\]

for every sample path, so

\[
X_n\xrightarrow{a.s.}0.
\]

In [ ]:
n_max = 200
paths = 6

U = rng.uniform(0, 1, size=(paths, n_max))
n_grid = np.arange(1, n_max+1)
X = U / n_grid

plt.figure(figsize=(7, 4))
for i in range(paths):
    plt.plot(n_grid, X[i], label=f"path {i+1}", alpha=0.8)
plt.xlabel("n")
plt.ylabel(r"$X_n=U_n/n$")
plt.title("Almost Sure Convergence Along Sample Paths")
plt.legend()
plt.show()

### Full Solution

For every outcome $\omega$,

\[
0\le X_n(\omega)=\frac{U_n(\omega)}{n}\le \frac1n.
\]

Since $1/n\to0$, the squeeze theorem gives

\[
X_n(\omega)\to0
\]

for every $\omega$. Therefore,

\[
P\left(\lim_{n\to\infty}X_n=0\right)=1.
\]

Thus

\[
X_n\xrightarrow{a.s.}0.
\]

## 6. Convergence in Mean Implies Convergence in Probability

If

\[
E|X_n-X|\to0,
\]

then by Markov's inequality,

\[
P(|X_n-X|\ge \epsilon)
\le
\frac{E|X_n-X|}{\epsilon}
\to0.
\]

Thus

\[
X_n\xrightarrow{L^1}X
\quad\Longrightarrow\quad
X_n\xrightarrow{P}X.
\]

We verify this with $X_n=U/n$.

In [ ]:
n_values = np.arange(1, 301)
eps = 0.01

# X_n = U/n
# E|X_n| = E[U]/n = 1/(2n)
mean_abs = 1/(2*n_values)

# P(U/n >= eps) = P(U >= n eps)
# = 1 - n eps if n eps < 1, otherwise 0
prob_tail = np.maximum(1 - n_values*eps, 0)

markov_bound = mean_abs / eps

plt.figure(figsize=(7, 4))
plt.plot(n_values, prob_tail, label=r"Exact $P(|X_n|\ge \epsilon)$")
plt.plot(n_values, np.minimum(markov_bound, 1), label="Markov bound")
plt.xlabel("n")
plt.ylabel("probability / bound")
plt.title(r"$L^1$ Convergence Implies Convergence in Probability")
plt.legend()
plt.show()

### Full Solution

For $X_n=U/n$,

\[
E|X_n|=\frac{E[U]}{n}=\frac{1}{2n}\to0.
\]

Therefore $X_n\to0$ in mean.

For any $\epsilon>0$,

\[
P(|X_n|\ge\epsilon)
=
P\left(U\ge n\epsilon\right).
\]

This goes to zero once $n\epsilon\ge1$. Hence $X_n\to0$ in probability as well.

## 7. Weak Law of Large Numbers

Let $X_1,\ldots,X_n$ be iid with

\[
E[X_i]=\mu,\qquad \operatorname{Var}(X_i)=\sigma^2<\infty.
\]

The Weak Law of Large Numbers says

\[
\bar X_n\xrightarrow{P}\mu.
\]

Using Chebyshev's inequality,

\[
P(|\bar X_n-\mu|\ge \epsilon)
\le
\frac{\sigma^2}{n\epsilon^2}.
\]

In [ ]:
# Bernoulli example
p = 0.4
mu = p
sigma2 = p*(1-p)
eps = 0.05

n_values = np.arange(10, 2001, 10)

# Exact tail probabilities
exact = []
for n in n_values:
    low = int(np.floor(n*(p-eps)))
    high = int(np.ceil(n*(p+eps)))
    p_low = stats.binom.cdf(low, n, p)
    p_high = 1 - stats.binom.cdf(high-1, n, p)
    exact.append(p_low + p_high)
exact = np.array(exact)

cheb = sigma2 / (n_values * eps**2)

plt.figure(figsize=(7, 4))
plt.plot(n_values, exact, label="Exact probability")
plt.plot(n_values, np.minimum(cheb, 1), label="Chebyshev bound")
plt.xlabel("n")
plt.ylabel(r"$P(|\bar X_n-p|\ge \epsilon)$")
plt.title("Weak Law of Large Numbers")
plt.legend()
plt.show()

### Full Solution

For Bernoulli$(p)$ variables,

\[
E[X_i]=p,\qquad \operatorname{Var}(X_i)=p(1-p).
\]

Thus

\[
E[\bar X_n]=p,
\qquad
\operatorname{Var}(\bar X_n)=\frac{p(1-p)}{n}.
\]

By Chebyshev,

\[
P(|\bar X_n-p|\ge\epsilon)
\le
\frac{p(1-p)}{n\epsilon^2}\to0.
\]

Therefore,

\[
\bar X_n\xrightarrow{P}p.
\]

## 8. Strong Law of Large Numbers

The Strong Law of Large Numbers says

\[
\bar X_n\xrightarrow{a.s.}\mu.
\]

This means that for almost every simulated path, the running average converges to the true mean.

We simulate many running averages for Bernoulli$(0.4)$ variables.

In [ ]:
paths = 8
n_max = 2000
p = 0.4

X = rng.binomial(1, p, size=(paths, n_max))
running_means = np.cumsum(X, axis=1) / np.arange(1, n_max+1)

plt.figure(figsize=(7, 4))
for i in range(paths):
    plt.plot(np.arange(1, n_max+1), running_means[i], alpha=0.8)
plt.axhline(p, linestyle="--", label="true mean p=0.4")
plt.xlabel("n")
plt.ylabel(r"$\bar X_n$")
plt.title("Strong Law of Large Numbers: Sample Paths")
plt.legend()
plt.show()

### Full Solution

The WLLN is a statement about probabilities:

\[
P(|\bar X_n-\mu|>\epsilon)\to0.
\]

The SLLN is stronger. It says that the full sequence of running averages converges along almost every sample path:

\[
P\left(\lim_{n\to\infty}\bar X_n=\mu\right)=1.
\]

The plot shows several sample paths moving toward $0.4$.

## 9. Central Limit Theorem

Let $X_1,\ldots,X_n$ be iid with mean $\mu$ and variance $\sigma^2<\infty$.

The Central Limit Theorem says

\[
\frac{\bar X_n-\mu}{\sigma/\sqrt n}
\xrightarrow{D}N(0,1).
\]

Equivalently,

\[
\bar X_n \approx N\left(\mu,\frac{\sigma^2}{n}\right)
\]

for large $n$.

In [ ]:
# CLT for exponential distribution
reps = 80_000
n_values = [2, 5, 10, 30, 100]

mu = 1.0
sigma = 1.0

for n in n_values:
    samples = rng.exponential(scale=1.0, size=(reps, n))
    z = (samples.mean(axis=1) - mu) / (sigma / np.sqrt(n))

    grid = np.linspace(-4, 4, 400)
    plt.figure(figsize=(7, 4))
    plt.hist(z, bins=70, density=True, alpha=0.7, label=f"standardized mean, n={n}")
    plt.plot(grid, stats.norm.pdf(grid), label="N(0,1)")
    plt.xlabel("z")
    plt.ylabel("Density")
    plt.title(f"CLT for Exponential Population, n={n}")
    plt.legend()
    plt.show()

### Full Solution

For $X_i\sim \mathrm{Exponential}(1)$,

\[
\mu=1,\qquad \sigma^2=1.
\]

The standardized sample mean is

\[
Z_n=\frac{\bar X_n-1}{1/\sqrt n}.
\]

The CLT says

\[
Z_n\xrightarrow{D}N(0,1).
\]

The histograms become closer to the standard normal curve as $n$ increases.

## 10. CLT for a Discrete Distribution

Suppose

\[
P(X=0)=0.2,\qquad P(X=1)=0.5,\qquad P(X=2)=0.3.
\]

Then

\[
\mu=E[X]=1.1,
\]

and

\[
\sigma^2=E[X^2]-\mu^2=1.7-1.21=0.49.
\]

For

\[
S_n=X_1+\cdots+X_n,
\]

the CLT gives

\[
S_n\approx N(n\mu,n\sigma^2).
\]

We approximate

\[
P(S_{50}\le 60)
\]

with and without continuity correction.

In [ ]:
values = np.array([0, 1, 2])
probs = np.array([0.2, 0.5, 0.3])

mu = np.sum(values * probs)
EX2 = np.sum(values**2 * probs)
sigma2 = EX2 - mu**2
sigma = np.sqrt(sigma2)

n = 50
k = 60

z_no_cc = (k - n*mu) / (sigma*np.sqrt(n))
approx_no_cc = stats.norm.cdf(z_no_cc)

z_cc = (k + 0.5 - n*mu) / (sigma*np.sqrt(n))
approx_cc = stats.norm.cdf(z_cc)

print("mu:", mu)
print("sigma^2:", sigma2)
print("No continuity correction z:", z_no_cc)
print("CLT approximation without correction:", approx_no_cc)
print("Continuity correction z:", z_cc)
print("CLT approximation with correction:", approx_cc)

# Simulation check
reps = 200_000
samples = rng.choice(values, size=(reps, n), p=probs)
S = samples.sum(axis=1)
sim_prob = np.mean(S <= k)

print("Simulation P(S_50 <= 60):", sim_prob)

### Full Solution

Compute

\[
E[X]=0(0.2)+1(0.5)+2(0.3)=1.1.
\]

Also,

\[
E[X^2]=0^2(0.2)+1^2(0.5)+2^2(0.3)=1.7.
\]

Therefore,

\[
\operatorname{Var}(X)=1.7-1.1^2=0.49.
\]

For $n=50$,

\[
E[S_{50}]=50(1.1)=55,
\qquad
\operatorname{SD}(S_{50})=\sqrt{50(0.49)}\approx 4.9497.
\]

Without continuity correction,

\[
P(S_{50}\le60)
\approx
\Phi\left(\frac{60-55}{4.9497}\right).
\]

With continuity correction,

\[
P(S_{50}\le60)
\approx
\Phi\left(\frac{60.5-55}{4.9497}\right).
\]

## 11. Continuous Mapping Theorem

If

\[
X_n\xrightarrow{P}X,
\]

and $g$ is continuous, then

\[
g(X_n)\xrightarrow{P}g(X).
\]

Similarly, if

\[
X_n\xrightarrow{D}X,
\]

then

\[
g(X_n)\xrightarrow{D}g(X).
\]

We illustrate with

\[
X_n=\bar X_n
\]

where $X_i\sim\mathrm{Uniform}(0,1)$, so $\bar X_n\xrightarrow{P}1/2$.  
Let $g(x)=x^2$.

In [ ]:
reps = 80_000
n_values = [5, 20, 100, 500]

rows = []
for n in n_values:
    samples = rng.uniform(0, 1, size=(reps, n))
    xbar = samples.mean(axis=1)
    gx = xbar**2
    rows.append([n, xbar.mean(), xbar.var(ddof=0), gx.mean(), gx.var(ddof=0)])

pd.DataFrame(rows, columns=[
    "n", "Mean of Xbar", "Var of Xbar",
    "Mean of Xbar^2", "Var of Xbar^2"
])

### Full Solution

By the WLLN,

\[
\bar X_n\xrightarrow{P}E[X_i]=\frac12.
\]

Since $g(x)=x^2$ is continuous,

\[
g(\bar X_n)=\bar X_n^2\xrightarrow{P}g(1/2)=\frac14.
\]

The simulation shows the variance of $\bar X_n^2$ shrinking and its mean approaching $0.25$.

## 12. Delta Method as a Continuous Mapping CLT Tool

The continuous mapping theorem gives convergence of transformations.  
The delta method gives the approximate distribution.

If

\[
\sqrt n(\bar X_n-\mu)\xrightarrow{D}N(0,\sigma^2),
\]

and $g$ is differentiable at $\mu$, then

\[
\sqrt n(g(\bar X_n)-g(\mu))
\xrightarrow{D}
N(0,[g'(\mu)]^2\sigma^2).
\]

Example: $X_i\sim \mathrm{Exponential}(1)$, $\mu=1$, $\sigma^2=1$, and $g(x)=\log x$.

In [ ]:
reps = 100_000
n = 80

samples = rng.exponential(scale=1.0, size=(reps, n))
xbar = samples.mean(axis=1)

# delta method: sqrt(n)(log(xbar)-log(1)) -> N(0, 1)
z_delta = np.sqrt(n) * (np.log(xbar) - 0)

grid = np.linspace(-4, 4, 400)

plt.figure(figsize=(7, 4))
plt.hist(z_delta, bins=70, density=True, alpha=0.7, label=r"$\sqrt n(\log \bar X_n-\log 1)$")
plt.plot(grid, stats.norm.pdf(grid), label="N(0,1) theory")
plt.xlabel("z")
plt.ylabel("Density")
plt.title("Delta Method Example")
plt.legend()
plt.show()

### Full Solution

For $X_i\sim\mathrm{Exponential}(1)$,

\[
\mu=1,\qquad \sigma^2=1.
\]

Let $g(x)=\log x$. Then

\[
g'(x)=\frac1x,
\qquad
g'(1)=1.
\]

Therefore,

\[
\sqrt n(\log \bar X_n-\log 1)
\xrightarrow{D}
N(0,1^2\cdot 1)
=
N(0,1).
\]

## 13. Slutsky's Theorem

If

\[
X_n\xrightarrow{D}X
\]

and

\[
Y_n\xrightarrow{P}c,
\]

then

\[
X_n+Y_n\xrightarrow{D}X+c,
\]

\[
X_nY_n\xrightarrow{D}cX,
\]

and, if $c\ne0$,

\[
\frac{X_n}{Y_n}\xrightarrow{D}\frac{X}{c}.
\]

This justifies replacing unknown $\sigma$ by consistent estimator $S$ in many statistics.

In [ ]:
# Demonstrate Slutsky using normal sample:
# Z_n = sqrt(n)(Xbar-mu)/sigma -> N(0,1)
# S/sigma -> 1 in probability
# T_n = sqrt(n)(Xbar-mu)/S -> N(0,1) asymptotically, exactly t_{n-1} under normality.

reps = 100_000
n_values = [5, 10, 30, 100]
mu = 0
sigma = 2

for n in n_values:
    samples = rng.normal(mu, sigma, size=(reps, n))
    xbar = samples.mean(axis=1)
    s = samples.std(axis=1, ddof=1)
    t_stat = np.sqrt(n) * (xbar - mu) / s

    grid = np.linspace(-4, 4, 400)
    plt.figure(figsize=(7, 4))
    plt.hist(t_stat, bins=70, density=True, alpha=0.7, label=f"self-normalized statistic, n={n}")
    plt.plot(grid, stats.norm.pdf(grid), label="N(0,1)")
    plt.xlabel("value")
    plt.ylabel("Density")
    plt.title(f"Slutsky: Replacing sigma by S, n={n}")
    plt.legend()
    plt.show()

### Full Solution

The CLT gives

\[
\frac{\sqrt n(\bar X_n-\mu)}{\sigma}\xrightarrow{D}N(0,1).
\]

Also, the sample standard deviation is consistent:

\[
S\xrightarrow{P}\sigma,
\]

so

\[
\frac{S}{\sigma}\xrightarrow{P}1.
\]

By Slutsky,

\[
\frac{\sqrt n(\bar X_n-\mu)}{S}
=
\frac{\sqrt n(\bar X_n-\mu)/\sigma}{S/\sigma}
\xrightarrow{D}
N(0,1).
\]

## 14. Concentration of Gaussian Means

If

\[
X_1,\ldots,X_n\sim N(0,\sigma^2),
\]

then

\[
\bar X_n\sim N\left(0,\frac{\sigma^2}{n}\right).
\]

A Gaussian concentration bound is

\[
P(|\bar X_n|>\epsilon)
\le
2\exp\left(-\frac{n\epsilon^2}{2\sigma^2}\right).
\]

In [ ]:
sigma = 1.0
eps = 0.2
n_values = np.arange(10, 1001, 10)

true_prob = 2 * (1 - stats.norm.cdf(eps * np.sqrt(n_values) / sigma))
gaussian_bound = 2 * np.exp(-n_values * eps**2 / (2*sigma**2))
cheb_bound = sigma**2 / (n_values * eps**2)

plt.figure(figsize=(7, 4))
plt.plot(n_values, true_prob, label="Exact Gaussian tail")
plt.plot(n_values, np.minimum(gaussian_bound, 1), label="Gaussian concentration bound")
plt.plot(n_values, np.minimum(cheb_bound, 1), label="Chebyshev bound")
plt.yscale("log")
plt.xlabel("n")
plt.ylabel("probability / bound")
plt.title("Concentration of a Gaussian Sample Mean")
plt.legend()
plt.show()

### Full Solution

Since

\[
\bar X_n\sim N\left(0,\frac{\sigma^2}{n}\right),
\]

we can write

\[
\frac{\sqrt n\bar X_n}{\sigma}\sim N(0,1).
\]

Thus

\[
P(|\bar X_n|>\epsilon)
=
P\left(|Z|>\frac{\sqrt n\epsilon}{\sigma}\right).
\]

The normal tail bound gives

\[
P(|Z|>t)\le 2e^{-t^2/2}.
\]

Substitute $t=\sqrt n\epsilon/\sigma$:

\[
P(|\bar X_n|>\epsilon)
\le
2e^{-n\epsilon^2/(2\sigma^2)}.
\]

## 15. Concentration of a Maximum

Let

\[
Z_n=\max\{|X_1|,\ldots,|X_n|\},
\]

where $X_i\sim N(0,\sigma^2)$ independently.

Using the union bound,

\[
P(Z_n>\epsilon)
\le
\sum_{i=1}^n P(|X_i|>\epsilon)
\le
2n\exp\left(-\frac{\epsilon^2}{2\sigma^2}\right).
\]

In [ ]:
sigma = 1.0
n = 100
eps_grid = np.linspace(1, 5, 200)

# Exact probability P(max |Xi| > eps) = 1 - P(|X| <= eps)^n
p_abs_le = 2*stats.norm.cdf(eps_grid/sigma) - 1
exact = 1 - p_abs_le**n

union_bound = 2*n*np.exp(-eps_grid**2/(2*sigma**2))

plt.figure(figsize=(7, 4))
plt.plot(eps_grid, exact, label="Exact probability")
plt.plot(eps_grid, np.minimum(union_bound, 1), label="Union/Gaussian tail bound")
plt.yscale("log")
plt.xlabel(r"$\epsilon$")
plt.ylabel("probability / bound")
plt.title(r"Concentration of $\max_i |X_i|$")
plt.legend()
plt.show()

### Full Solution

Let

\[
A_i=\{|X_i|>\epsilon\}.
\]

Then

\[
\{Z_n>\epsilon\}=\bigcup_{i=1}^n A_i.
\]

By Boole's inequality,

\[
P(Z_n>\epsilon)
\le
\sum_{i=1}^n P(A_i)
=
nP(|X_1|>\epsilon).
\]

Using

\[
P(|X_1|>\epsilon)\le 2e^{-\epsilon^2/(2\sigma^2)},
\]

we obtain

\[
P(Z_n>\epsilon)
\le
2n e^{-\epsilon^2/(2\sigma^2)}.
\]

## 16. Chebyshev vs Hoeffding for Sample Means

If $X_i$ are iid with variance $\sigma^2$, Chebyshev gives

\[
P(|\bar X_n-E\bar X_n|\ge\epsilon)
\le
\frac{\sigma^2}{n\epsilon^2}.
\]

If $0\le X_i\le1$, Hoeffding gives

\[
P(|\bar X_n-E\bar X_n|\ge\epsilon)
\le
2e^{-2n\epsilon^2}.
\]

Hoeffding has exponential decay, while Chebyshev has polynomial decay.

In [ ]:
p = 0.4
sigma2 = p*(1-p)
eps = 0.05

n_values = np.arange(10, 3001, 10)

cheb = sigma2 / (n_values * eps**2)
hoeff = 2*np.exp(-2*n_values*eps**2)

# exact for Bernoulli
exact = []
for n in n_values:
    low = int(np.floor(n*(p-eps)))
    high = int(np.ceil(n*(p+eps)))
    p_low = stats.binom.cdf(low, n, p)
    p_high = 1 - stats.binom.cdf(high-1, n, p)
    exact.append(p_low + p_high)
exact = np.array(exact)

plt.figure(figsize=(7, 4))
plt.plot(n_values, exact, label="Exact")
plt.plot(n_values, np.minimum(cheb, 1), label="Chebyshev")
plt.plot(n_values, np.minimum(hoeff, 1), label="Hoeffding")
plt.yscale("log")
plt.xlabel("n")
plt.ylabel(r"$P(|\bar X_n-p|\ge\epsilon)$")
plt.title("Concentration: Chebyshev vs Hoeffding")
plt.legend()
plt.show()

### Full Solution

For Bernoulli$(p)$ variables,

\[
\sigma^2=p(1-p).
\]

Chebyshev gives

\[
P(|\bar X_n-p|\ge\epsilon)
\le
\frac{p(1-p)}{n\epsilon^2}.
\]

Hoeffding applies because $0\le X_i\le1$:

\[
P(|\bar X_n-p|\ge\epsilon)
\le
2e^{-2n\epsilon^2}.
\]

For large $n$, exponential decay is much faster than $1/n$ decay.

## 17. High-Dimensional Proportion Estimation

Suppose we estimate $d$ proportions

\[
p_1,\ldots,p_d
\]

using independent Bernoulli samples of size $n$ for each coordinate. Let

\[
\hat p_j=\frac1n\sum_{i=1}^n X_{ij}.
\]

By Hoeffding and the union bound,

\[
P\left(\max_{1\le j\le d}|\hat p_j-p_j|\ge\epsilon\right)
\le
2d e^{-2n\epsilon^2}.
\]

To make this at most $\delta$, it is enough to choose

\[
n\ge \frac{\log(2d/\delta)}{2\epsilon^2}.
\]

In [ ]:
d = 1000
eps = 0.05
delta = 0.05

n_required = int(np.ceil(np.log(2*d/delta) / (2*eps**2)))
print("Required n from Hoeffding + union bound:", n_required)

# Simulate once with random true probabilities
p_vec = rng.uniform(0.1, 0.9, size=d)
X = rng.binomial(1, p_vec, size=(n_required, d))
p_hat = X.mean(axis=0)
max_error = np.max(np.abs(p_hat - p_vec))

print("Observed max error in one simulation:", max_error)
print("epsilon:", eps)

In [ ]:
# Repeat experiment to estimate probability of max error > epsilon
reps = 300
bad = 0

for _ in range(reps):
    p_vec = rng.uniform(0.1, 0.9, size=d)
    X = rng.binomial(1, p_vec, size=(n_required, d))
    p_hat = X.mean(axis=0)
    bad += (np.max(np.abs(p_hat - p_vec)) >= eps)

empirical_bad_prob = bad / reps
bound = 2*d*np.exp(-2*n_required*eps**2)

print("Empirical P(max error >= eps):", empirical_bad_prob)
print("Hoeffding-union bound:", bound)
print("Target delta:", delta)

### Full Solution

For a fixed coordinate $j$, Hoeffding gives

\[
P(|\hat p_j-p_j|\ge\epsilon)
\le
2e^{-2n\epsilon^2}.
\]

Now apply the union bound over $d$ coordinates:

\[
P\left(\max_{1\le j\le d}|\hat p_j-p_j|\ge\epsilon\right)
=
P\left(\bigcup_{j=1}^d \{|\hat p_j-p_j|\ge\epsilon\}\right)
\le
\sum_{j=1}^d 2e^{-2n\epsilon^2}
=
2d e^{-2n\epsilon^2}.
\]

To make this less than $\delta$,

\[
2d e^{-2n\epsilon^2}\le\delta.
\]

Solving gives

\[
n\ge
\frac{\log(2d/\delta)}{2\epsilon^2}.
\]

# Practice Problems with Full Solutions

## Practice Problem 1 — Convergence in Probability

Let

\[
X_n=
\begin{cases}
1, & \text{with probability }1/n,\\
0, & \text{with probability }1-1/n.
\end{cases}
\]

Show that $X_n\xrightarrow{P}0$.

In [ ]:
n_values = np.arange(1, 501)
prob_tail = 1/n_values

plt.figure(figsize=(7, 4))
plt.plot(n_values, prob_tail)
plt.xlabel("n")
plt.ylabel(r"$P(|X_n|>0.5)$")
plt.title("Practice 1: Tail Probability Goes to Zero")
plt.show()

### Solution

For $\epsilon=0.5$,

\[
P(|X_n-0|>\epsilon)=P(X_n=1)=\frac1n\to0.
\]

For any $0<\epsilon<1$, the same calculation holds.  
For $\epsilon\ge1$, the probability is $0$.

Thus for every $\epsilon>0$,

\[
P(|X_n-0|>\epsilon)\to0.
\]

Therefore,

\[
X_n\xrightarrow{P}0.
\]

## Practice Problem 2 — Convergence in Mean

For the same $X_n$,

\[
X_n=
\begin{cases}
1, & \text{with probability }1/n,\\
0, & \text{with probability }1-1/n,
\end{cases}
\]

show that $X_n\to0$ in mean.

In [ ]:
n_values = np.arange(1, 501)
mean_abs = 1/n_values

plt.figure(figsize=(7, 4))
plt.plot(n_values, mean_abs)
plt.xlabel("n")
plt.ylabel(r"$E|X_n|$")
plt.title("Practice 2: Convergence in Mean")
plt.show()

### Solution

Since $X_n\ge0$,

\[
E|X_n-0|=E[X_n].
\]

Compute

\[
E[X_n]=1\cdot\frac1n+0\cdot\left(1-\frac1n\right)=\frac1n\to0.
\]

Therefore,

\[
X_n\xrightarrow{L^1}0.
\]

## Practice Problem 3 — CLT Approximation

Let

\[
X_1,\ldots,X_{100}\sim \mathrm{Bernoulli}(0.3).
\]

Approximate

\[
P(\bar X_{100}\le0.35)
\]

using the CLT.

In [ ]:
n = 100
p = 0.3
q = 0.35

mu = p
sigma = np.sqrt(p*(1-p))
z = (q - mu) / (sigma/np.sqrt(n))
approx = stats.norm.cdf(z)

# Exact probability: S <= 35
exact = stats.binom.cdf(35, n, p)

print("z:", z)
print("CLT approximation:", approx)
print("Exact binomial probability:", exact)

### Solution

For Bernoulli$(p)$,

\[
\mu=p=0.3,
\qquad
\sigma^2=p(1-p)=0.3(0.7)=0.21.
\]

By the CLT,

\[
\frac{\bar X_{100}-0.3}{\sqrt{0.21/100}}
\approx N(0,1).
\]

Therefore,

\[
P(\bar X_{100}\le0.35)
\approx
\Phi\left(
\frac{0.35-0.3}{\sqrt{0.21/100}}
\right).
\]

Python computes the numerical value.

## Practice Problem 4 — Continuity Correction

Use a continuity correction for

\[
P(\bar X_{100}\le0.35)
\]

when $X_i\sim\mathrm{Bernoulli}(0.3)$.

Hint:

\[
\bar X_{100}\le0.35
\iff
S_{100}\le35.
\]

Use $35.5$ in the normal approximation.

In [ ]:
n = 100
p = 0.3
k = 35

mu_S = n*p
sd_S = np.sqrt(n*p*(1-p))

z_cc = (k + 0.5 - mu_S) / sd_S
approx_cc = stats.norm.cdf(z_cc)
exact = stats.binom.cdf(k, n, p)

print("z with continuity correction:", z_cc)
print("CLT approximation with correction:", approx_cc)
print("Exact binomial probability:", exact)

### Solution

Let

\[
S_{100}=X_1+\cdots+X_{100}.
\]

Then

\[
S_{100}\sim \mathrm{Binomial}(100,0.3),
\]

and

\[
\bar X_{100}\le0.35
\iff
S_{100}\le35.
\]

The CLT approximates $S_{100}$ by

\[
N(np,np(1-p)).
\]

With continuity correction,

\[
P(S_{100}\le35)
\approx
\Phi\left(
\frac{35.5-30}{\sqrt{21}}
\right).
\]

## Practice Problem 5 — Continuous Mapping Theorem

Suppose

\[
X_n\xrightarrow{P}2.
\]

Find the probability limit of

\[
Y_n=3X_n^2+1.
\]

In [ ]:
limit_X = 2
limit_Y = 3*limit_X**2 + 1
print("Limit of Y_n:", limit_Y)

### Solution

Let

\[
g(x)=3x^2+1.
\]

This function is continuous. By the continuous mapping theorem,

\[
g(X_n)\xrightarrow{P}g(2).
\]

Therefore,

\[
Y_n=3X_n^2+1\xrightarrow{P}3(2)^2+1=13.
\]

## Practice Problem 6 — Slutsky's Theorem

Suppose

\[
X_n\xrightarrow{D}N(0,1),
\qquad
Y_n\xrightarrow{P}2.
\]

Find the limiting distribution of

\[
Z_n=2X_n+Y_n.
\]

In [ ]:
# Simulate a Slutsky-style example:
# X_n ~ approximately N(0,1), Y_n = 2 + noise/n
reps = 100_000
n = 500

Xn = rng.normal(0, 1, size=reps)
Yn = 2 + rng.normal(0, 1, size=reps)/n
Zn = 2*Xn + Yn

grid = np.linspace(-6, 10, 400)
theory_pdf = stats.norm.pdf(grid, loc=2, scale=2)

plt.figure(figsize=(7, 4))
plt.hist(Zn, bins=70, density=True, alpha=0.7, label="Simulation")
plt.plot(grid, theory_pdf, label=r"$N(2,4)$ theory")
plt.xlabel("z")
plt.ylabel("Density")
plt.title("Practice 6: Slutsky's Theorem")
plt.legend()
plt.show()

### Solution

Since

\[
X_n\xrightarrow{D}X,\qquad X\sim N(0,1),
\]

and

\[
Y_n\xrightarrow{P}2,
\]

Slutsky's theorem gives

\[
2X_n+Y_n\xrightarrow{D}2X+2.
\]

Since $X\sim N(0,1)$,

\[
2X+2\sim N(2,4).
\]

Thus

\[
Z_n\xrightarrow{D}N(2,4).
\]

## Practice Problem 7 — Hoeffding Sample Size

For Bernoulli samples, find $n$ such that

\[
P(|\bar X_n-p|\ge0.02)\le0.01
\]

using Hoeffding's inequality.

In [ ]:
eps = 0.02
delta = 0.01

n_required = int(np.ceil(np.log(2/delta)/(2*eps**2)))
bound = 2*np.exp(-2*n_required*eps**2)

print("Required n:", n_required)
print("Hoeffding bound at this n:", bound)

### Solution

Hoeffding gives

\[
P(|\bar X_n-p|\ge\epsilon)
\le
2e^{-2n\epsilon^2}.
\]

We want

\[
2e^{-2n(0.02)^2}\le0.01.
\]

Solving,

\[
n\ge
\frac{\log(2/0.01)}{2(0.02)^2}.
\]

Python computes the ceiling of this value.

## Practice Problem 8 — High-Dimensional Estimation

Suppose $d=5000$ proportions must be estimated simultaneously.  
Find $n$ such that

\[
P\left(\max_{1\le j\le d}|\hat p_j-p_j|\ge0.03\right)\le0.05.
\]

Use Hoeffding plus union bound.

In [ ]:
d = 5000
eps = 0.03
delta = 0.05

n_required = int(np.ceil(np.log(2*d/delta)/(2*eps**2)))
print("Required n:", n_required)

bound = 2*d*np.exp(-2*n_required*eps**2)
print("Bound:", bound)

### Solution

By Hoeffding and the union bound,

\[
P\left(\max_{1\le j\le d}|\hat p_j-p_j|\ge\epsilon\right)
\le
2d e^{-2n\epsilon^2}.
\]

Set this less than $\delta$:

\[
2d e^{-2n\epsilon^2}\le\delta.
\]

Solving,

\[
n\ge
\frac{\log(2d/\delta)}{2\epsilon^2}.
\]

With $d=5000$, $\epsilon=0.03$, and $\delta=0.05$, Python gives the required sample size.

# Summary

In this lab, we studied the computational meaning of Section 9:

- Convergence in probability
- Convergence in distribution
- Convergence in mean
- Almost sure convergence
- WLLN and SLLN
- CLT and normal approximations
- Continuity corrections
- Continuous Mapping Theorem
- Delta method
- Slutsky's theorem
- Gaussian concentration
- Concentration of maxima
- Chebyshev versus Hoeffding
- High-dimensional estimation rates

The central message is:

\[
\boxed{
\text{Limit theorems explain why statistical procedures become stable and approximately normal.}
}
\]